# Pipeline completo — IA Generativa y Saber Pro (2021-2024)

**Investigación:** *Disparidades en el desempeño Saber Pro y su asociación con el período de adopción de IA Generativa.*
Eduardo Andrés Victoria Cadena — Universidad Surcolombiana, 2026.

Este cuaderno ejecuta de extremo a extremo:
1. **Preparación** de microdatos DataICFES → `df_consolidado.csv`.
2. **Parte 1** — análisis bivariado / descriptivo (Tabla 3 + 4 figuras).
3. **Parte 2** — 18 modelos MCO + diagnósticos + corrección por pruebas múltiples.

### Antes de ejecutar

1. En **Mi unidad** de Google Drive cree la carpeta `IA_EDUCACION_SUPERIOR`.
2. Coloque dentro los cuatro archivos crudos:
   - `Examen_Saber_Pro_Genericas_2021.txt`
   - `Examen_Saber_Pro_Genericas_2022.txt`
   - `Examen_Saber_Pro_Genericas_2023.txt`
   - `Examen_Saber_Pro_Genericas_2024.txt`
3. Asegúrese de que los archivos `preparar_datos.py`, `analisis_descriptivo.py`, `regresion_mco.py` y `main.py` estén en `Mi unidad/IA_EDUCACION_SUPERIOR/python/` (o en `/content/` si los subió manualmente).
4. Ejecute las celdas en orden.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUTA_PROYECTO = '/content/drive/MyDrive/IA_EDUCACION_SUPERIOR'
print('Ruta del proyecto:', RUTA_PROYECTO)

## 2. Hacer accesibles los módulos
Copia los `.py` desde Drive al directorio de trabajo de Colab y los agrega al `sys.path` para poder importarlos. Si Drive no contiene la carpeta `python/`, esta celda no falla (asume que ya tienes los `.py` en `/content/`).

In [ ]:
import os, sys, shutil

DESTINO = '/content'
ORIGEN = os.path.join(RUTA_PROYECTO, 'python')

if os.path.isdir(ORIGEN):
    for nombre in ('preparar_datos.py', 'analisis_descriptivo.py',
                   'regresion_mco.py', 'main.py'):
        src = os.path.join(ORIGEN, nombre)
        if os.path.exists(src):
            shutil.copy(src, DESTINO)
            print(' Copiado:', nombre)
else:
    print('No se encontró', ORIGEN, '— se asume que los .py ya están en /content/')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

## 3. Verificar dependencias
Colab ya incluye `pandas`, `numpy`, `scipy`, `statsmodels` y `matplotlib`. Si por alguna razón faltase alguno, los módulos los instalarán automáticamente al importarlos.

In [ ]:
import importlib
for pkg in ['pandas', 'numpy', 'scipy', 'statsmodels', 'matplotlib']:
    ok = importlib.util.find_spec(pkg) is not None
    print(f'  {pkg:<12} {"OK" if ok else "FALTA"}')

## 4. Parte 0 — Preparación de datos
Carga los 4 `.txt` desde Drive, construye las 22 variables del estudio (incluyendo `cod_ies`, `municipio_ies` y `tipo_municipio` para los efectos fijos), aplica limpieza integral y produce `df_2021…df_2024.csv` + `df_consolidado.csv`.

In [ ]:
from preparar_datos import ejecutar_pipeline

dfs_anio, df_consolidado = ejecutar_pipeline()  # usa la ruta por defecto de Colab
df_2021, df_2022, df_2023, df_2024 = (dfs_anio[a] for a in (2021, 2022, 2023, 2024))
df_consolidado.head()

In [ ]:
# Distribución por cohorte y forma del dataframe consolidado.
print('df_consolidado:', df_consolidado.shape)
df_consolidado.groupby('periodo_ia')['puntaje_saberpro_generico'].agg(['count', 'mean', 'std']).round(2)

## 5. Parte 1 — Análisis bivariado / descriptivo
Construye la Tabla 3 (t-Welch + Mann-Whitney + χ²), la tabla de medias por departamento y genera las 4 figuras (boxplot por cohorte, boxplot por departamento, histograma comparativo y dispersión distancia ↔ puntaje).

In [ ]:
from analisis_descriptivo import ejecutar_analisis_descriptivo

resultado_descriptivo = ejecutar_analisis_descriptivo()
resultado_descriptivo['tabla3']

In [ ]:
# Visualizar las figuras generadas
from IPython.display import Image, display

dir_figs = os.path.join(RUTA_PROYECTO, 'procesados', 'figuras')
for fig in ('fig_boxplot_periodo.png',
            'fig_boxplot_departamento.png',
            'fig_histograma_cohortes.png',
            'fig_dispersion_distancia.png'):
    ruta = os.path.join(dir_figs, fig)
    if os.path.exists(ruta):
        print('\n', fig)
        display(Image(ruta))

## 6. Parte 2 — Regresión lineal múltiple por MCO
Estima 18 modelos (6 dependientes × 3 especificaciones: base, EF IES, EF tipo de municipio) con errores estándar clusterizados por IES, calcula los 5 diagnósticos para cada uno, evalúa el triángulo de colinealidad geográfica y corrige por pruebas múltiples (Holm + Benjamini-Hochberg).

In [ ]:
from regresion_mco import ejecutar_regresion

resultado_regresion = ejecutar_regresion()
resultado_regresion['beta_ia']

In [ ]:
# Triángulo de colinealidad geográfica (sólo agregado).
resultado_regresion['colinealidad']

In [ ]:
# Diagnósticos por modelo.
resultado_regresion['diagnosticos']

In [ ]:
# Tabla 4 — coeficientes de todos los modelos. Filtramos para ver solo β_IA.
tabla4 = resultado_regresion['tabla4']
tabla4[tabla4['termino'] == 'periodo_ia']

## 7. Archivos generados en Drive
Todos los resultados quedan persistidos en `Mi unidad/IA_EDUCACION_SUPERIOR/procesados/`:

```
procesados/
├── df_2021.csv  df_2022.csv  df_2023.csv  df_2024.csv
├── df_consolidado.csv
├── resultados/
│   ├── tabla3_descriptivo.csv          (Parte 1)
│   ├── tabla3_por_departamento.csv
│   ├── tabla4_coeficientes.csv         (Parte 2)
│   ├── diagnosticos.csv
│   ├── beta_ia_resumen.csv
│   └── colinealidad_geografica.csv
└── figuras/
    ├── fig_boxplot_periodo.png
    ├── fig_boxplot_departamento.png
    ├── fig_histograma_cohortes.png
    └── fig_dispersion_distancia.png
```

In [ ]:
# Listado de archivos generados.
import os
base = os.path.join(RUTA_PROYECTO, 'procesados')
for raiz, _, archivos in os.walk(base):
    for f in sorted(archivos):
        ruta = os.path.join(raiz, f)
        kb = os.path.getsize(ruta) / 1024
        print(f'  {ruta.replace(base + "/", ""):<55} {kb:>8.1f} KB')

## Alternativa: ejecutar todo en una sola celda
Si prefiere lanzar las tres partes con una sola llamada:

```python
from main import ejecutar_todo
ejecutar_todo()   # equivale a las tres celdas anteriores
```